In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

from skimage.filters import frangi
from sklearn.metrics import accuracy_score, f1_score, jaccard_score, confusion_matrix

def set_seed(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

use_cuda = torch.cuda.is_available()
features = [64, 128, 256, 512]
num_epochs = 40

if use_cuda:
    try:
        free_mem = torch.cuda.mem_get_info()[0] / (1024**3)
        if free_mem < 1.0:
            print(f"CUDA is available but only {free_mem:.2f} GB free VRAM. Falling back to CPU to avoid OOM.")
            use_cuda = False
        else:
            x_test = torch.zeros((1, 3, 256, 256), device="cuda")
            del x_test
            torch.cuda.empty_cache()
    except Exception as e:
        print(f"CUDA is available but testing allocation failed: {e}. Falling back to CPU.")
        use_cuda = False

device = torch.device("cuda" if use_cuda else "cpu")
print(f"Using device: {device}")

if device.type == "cpu":
    print("Using a lighter U-Net architecture and fewer epochs for faster CPU execution.")
    features = [32, 64, 128, 256]
    num_epochs = 20
else:
    print("GPU Model:", torch.cuda.get_device_name(0))

CUDA is available but testing allocation failed: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
. Falling back to CPU.
Using device: cpu
Using a lighter U-Net architecture and fewer epochs for faster CPU execution.


/home/clauss/projects/university/informatyka-w-medycynie-oko/.venv/lib/python3.14/site-packages/torch/cuda/__init__.py:384: UserWarning: Found GPU0 NVIDIA GeForce GTX 1050 Ti which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/clauss/projects/university/informatyka-w-medycynie-oko/.venv/lib/python3.14/site-packages/torch/cuda/__init__.py:502: UserWarning: 
NVIDIA GeForce GTX 1050 Ti wit

In [ ]:
class RetinalDataset(Dataset):
    def __init__(self, image_dir, label_dir, image_ids, transform=False, patch_size=(256, 256), val_padding=(2, 1, 2, 2)):
        self.image_ids = image_ids
        self.transform = transform
        self.patch_size = patch_size
        self.val_padding = val_padding
        self.images = []
        self.labels = []
        
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        
        print(f"Precomputing features for {len(image_ids)} images (transform={transform})...")
        for img_id in image_ids:
            img_path = Path(image_dir) / f"{img_id}.ppm"
            lbl_path = Path(label_dir) / f"{img_id}.vk.ppm"
            
            image_np = np.array(Image.open(img_path).convert("RGB"))
            label_np = np.array(Image.open(lbl_path).convert("L"))
            
            green = image_np[:, :, 1]
            enhanced_green = clahe.apply(green)
            
            enhanced_green_norm = enhanced_green.astype(float) / 255.0
            frangi_map = frangi(enhanced_green_norm, sigmas=(0.1, 0.5, 1.0, 1.5), black_ridges=True)
            
            if frangi_map.max() > 0:
                frangi_map = frangi_map / frangi_map.max()
            
            stacked = np.stack([
                enhanced_green.astype(np.float32) / 255.0,
                frangi_map.astype(np.float32),
                green.astype(np.float32) / 255.0
            ], axis=-1)
            
            self.images.append(stacked)
            self.labels.append(label_np)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_np = self.images[idx]
        lbl_np = self.labels[idx]
        
        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1) # 3 x H x W
        lbl_tensor = torch.from_numpy(lbl_np).unsqueeze(0).float() / 255.0 # 1 x H x W
        
        if self.transform:
            th, tw = self.patch_size
            _, h, w = img_tensor.shape
            top = random.randint(0, h - th)
            left = random.randint(0, w - tw)
            
            img_crop = img_tensor[:, top:top+th, left:left+tw]
            lbl_crop = lbl_tensor[:, top:top+th, left:left+tw]
            
            if random.random() > 0.5:
                img_crop = torch.flip(img_crop, dims=[2])
                lbl_crop = torch.flip(lbl_crop, dims=[2])
            if random.random() > 0.5:
                img_crop = torch.flip(img_crop, dims=[1])
                lbl_crop = torch.flip(lbl_crop, dims=[1])
            if random.random() > 0.5:
                angle = random.randint(-30, 30)
                img_crop = TF.rotate(img_crop, angle)
                lbl_crop = TF.rotate(lbl_crop, angle)
                
            return img_crop, (lbl_crop > 0.5).float()
        else:
            img_pad = TF.pad(img_tensor, self.val_padding, fill=0)
            lbl_pad = TF.pad(lbl_tensor, self.val_padding, fill=0)
            return img_pad, (lbl_pad > 0.5).float()

In [ ]:
DATA_DIR = Path("data")
ORIGINAL_DIR = DATA_DIR / "original"
LABELS_DIR = DATA_DIR / "labels"

all_image_ids = sorted([p.stem for p in ORIGINAL_DIR.glob("*.ppm")])
print(f"Found {len(all_image_ids)} images in total.")

train_ids = all_image_ids[:15]
val_ids = all_image_ids[15:]

train_dataset = RetinalDataset(ORIGINAL_DIR, LABELS_DIR, train_ids, transform=True)
val_dataset = RetinalDataset(ORIGINAL_DIR, LABELS_DIR, val_ids, transform=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

Found 20 images in total.
Precomputing features for 15 images (transform=True)...


Precomputing features for 5 images (transform=False)...


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.relu = nn.ReLU(inplace=True)
        self.skip = nn.Identity() if in_channels == out_channels else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        return self.relu(self.conv(x) + self.skip(x))

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.attns = nn.ModuleList()
        self.pools = nn.ModuleList()

        curr_in = in_channels
        for feature in features:
            self.downs.append(DoubleConv(curr_in, feature))
            self.pools.append(nn.Conv2d(feature, feature, kernel_size=2, stride=2))
            curr_in = feature

        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)
            )
            self.attns.append(
                AttentionGate(F_g=feature, F_l=feature, F_int=max(16, feature//2))
            )
            self.ups.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for idx, down in enumerate(self.downs):
            x = down(x)
            skip_connections.append(x)
            x = self.pools[idx](x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]
            
            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])
            
            attn_skip = self.attns[idx//2](g=x, x=skip_connection)
            
            concat_x = torch.cat((attn_skip, x), dim=1)
            x = self.ups[idx+1](concat_x)

        return self.final_conv(x)

In [6]:
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.1, beta=0.9, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        preds = preds.view(-1)
        targets = targets.view(-1)
        
        TP = (preds * targets).sum()
        FP = (preds * (1.0 - targets)).sum()
        FN = ((1.0 - preds) * targets).sum()
        
        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        return 1.0 - tversky

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

class FocalTverskyLoss(nn.Module):
    def __init__(self, focal_weight=0.4, tversky_weight=0.6, alpha=0.05, beta=0.95):
        super().__init__()
        self.focal = FocalLoss(alpha=0.25, gamma=2.0)
        self.tversky = TverskyLoss(alpha=alpha, beta=beta)
        self.focal_weight = focal_weight
        self.tversky_weight = tversky_weight

    def forward(self, preds, targets):
        return self.focal_weight * self.focal(preds, targets) + self.tversky_weight * self.tversky(preds, targets)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * images.size(0)
    return epoch_loss / len(loader.dataset)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    all_probs = []
    all_masks = []
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            epoch_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(outputs)
            
            # Crop padding (704x608 to 700x605)
            probs_cropped = probs[:, :, 1:606, 2:702]
            masks_cropped = masks[:, :, 1:606, 2:702]
            
            all_probs.append(probs_cropped.cpu())
            all_masks.append(masks_cropped.cpu())
            
    all_probs = torch.cat(all_probs, dim=0).numpy().ravel()
    all_masks = torch.cat(all_masks, dim=0).numpy().ravel()
    
    # Automated threshold sweep for best validation F1
    thresholds = np.arange(0.1, 0.9, 0.05)
    best_f1 = -1.0
    best_thresh = 0.5
    
    for thresh in thresholds:
        preds = (all_probs > thresh).astype(np.uint8)
        f1 = f1_score(all_masks, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
            
    # Compute metrics at best threshold
    best_preds = (all_probs > best_thresh).astype(np.uint8)
    acc = accuracy_score(all_masks, best_preds)
    
    tn, fp, fn, tp = confusion_matrix(all_masks, best_preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    return {
        "loss": epoch_loss / len(loader.dataset),
        "accuracy": acc,
        "f1": best_f1,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "best_threshold": best_thresh
    }

In [ ]:
model = AttentionUNet(in_channels=3, out_channels=1, features=features).to(device)
criterion = FocalTverskyLoss(focal_weight=0.4, tversky_weight=0.6, alpha=0.05, beta=0.95)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

history = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
    "val_sensitivity": [],
    "val_specificity": [],
    "best_threshold": []
}

best_val_f1 = 0.0
checkpoint_path = "best_unet_small_vessel.pth"

print(f"Starting Attention U-Net Training for {num_epochs} epochs...")
for epoch in range(1, num_epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = validate_epoch(model, val_loader, criterion, device)
    
    scheduler.step(val_metrics["loss"])
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_sensitivity"].append(val_metrics["sensitivity"])
    history["val_specificity"].append(val_metrics["specificity"])
    history["best_threshold"].append(val_metrics["best_threshold"])
    
    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Epoch {epoch:02d}: New best Val F1 = {best_val_f1:.4f} @ Threshold {val_metrics['best_threshold']:.2f}! Model saved.")
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} | Val F1: {val_metrics['f1']:.4f} (Thresh: {val_metrics['best_threshold']:.2f})")

Starting Attention U-Net Training for 20 epochs...


KeyboardInterrupt: 

In [ ]:
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path))
    print("Loaded best model checkpoint.")

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", color="royalblue", lw=2)
plt.plot(history["val_loss"], label="Val Loss", color="darkorange", lw=2)
plt.title("BCE + Tversky Loss Curves", fontsize=12)
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Loss", fontsize=10)
plt.grid(True, ls="--")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["val_iou"], label="Val IoU", color="forestgreen", lw=2)
plt.plot(history["val_f1"], label="Val F1 (Dice)", color="crimson", lw=2)
plt.plot(history["val_acc"], label="Val Accuracy", color="purple", lw=2)
plt.title("Validation Metrics", fontsize=12)
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Metric Value", fontsize=10)
plt.grid(True, ls="--")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
best_threshold = history["best_threshold"][np.argmax(history["val_f1"])]
print(f"Using Best Threshold for Visualization: {best_threshold:.2f}")


with torch.no_grad():
    for i in range(len(val_dataset)):
        img, mask = val_dataset[i]
        img_id = val_ids[i]
        
        img_tensor = img.unsqueeze(0).to(device)
        outputs = model(img_tensor)
        prob_map_full = torch.sigmoid(outputs).squeeze(0).squeeze(0).cpu().numpy()
        pred_mask_full = (prob_map_full > best_threshold).astype(np.uint8)
        
        prob_map = prob_map_full[1:606, 2:702]
        pred_mask = pred_mask_full[1:606, 2:702]
        
        img_np = img.permute(1, 2, 0).numpy()[1:606, 2:702, :]
        mask_np = mask.squeeze(0).numpy()[1:606, 2:702]
        
        flat_mask = mask_np.ravel()
        flat_pred = pred_mask.ravel()
        iou = jaccard_score(flat_mask, flat_pred, zero_division=0)
        f1 = f1_score(flat_mask, flat_pred, zero_division=0)
        
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Validation Sample: {img_id} | IoU = {iou:.4f} | F1 = {f1:.4f} (Thresh: {best_threshold:.2f})", fontsize=14)
        
        axes[0].imshow(img_np[:, :, 0], cmap="gray")
        axes[0].set_title("CLAHE Green Input")
        axes[0].axis("off")
        
        axes[1].imshow(mask_np, cmap="gray")
        axes[1].set_title("Ground Truth")
        axes[1].axis("off")
        
        pos = axes[2].imshow(prob_map, cmap="hot", vmin=0, vmax=1)
        axes[2].set_title("U-Net Probability Map")
        axes[2].axis("off")
        fig.colorbar(pos, ax=axes[2], fraction=0.046, pad=0.04)
        
        axes[3].imshow(pred_mask, cmap="gray")
        axes[3].set_title("Predicted Binary Mask")
        axes[3].axis("off")
        
        plt.tight_layout()
        plt.show()